In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.io import load_pkg_yaml

schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])
filename = '03_derived_analysis_dataset.parquet'
df = pd.read_parquet(DATA_DIR / filename)

In [3]:
print(df.columns.tolist())

['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarello_hx_recent_hosp_90d', 'tumbarello_hx_recent_ab

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
from tableone import TableOne

# 1. Load your master dataset
schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])
filename = '03_derived_analysis_dataset.parquet'
df_master = pd.read_parquet(DATA_DIR / filename)

# 1. Ensure auxiliary mockup columns exist if not already present
if "esbl_label" not in df_master.columns:
    df_master["esbl_label"] = np.random.choice(["Positive", "Negative"], size=len(df_master), p=[0.15, 0.85])


import pandas as pd

def convert_dtypes_and_classify(
    df,
    force_categorical=None,
    force_numerical=None
):
    """
    Explicitly casts columns and partitions them into categorical vs numerical features
    for Table 1, with optional manual override lists.
    """
    df = df.copy()

    # Default empty sets/lists if none provided
    force_categorical = set(force_categorical or [])
    force_numerical = set(force_numerical or [])

    # Define columns to exclude from the statistical summary table
    exclude_cols = {
        "SUBJECT", "SPELL_IDENTIFIER", "ENCNTR_ID", "EPISODE_IDENTIFIER",
        "ADMISSION_DATE", "ADMISSION_TIME", "DISCHARGE_DATE", "DEATH_DATE",
        "BIRTH_DATE", "UPDATE_DT_TM", "index_admission",
        "MAIN_SPECIALTY_CODE", "GENDER", "ETHNICITY", "definitions",
        "closest_culture_date", "esbl_status", "esbl_label"
    }

    # Candidate features for summary
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    categorical_cols = []
    numerical_cols = []

    for col in feature_cols:
        # --- 1. Manual Overrides Take Absolute Priority ---
        if col in force_categorical:
            categorical_cols.append(col)
            df[col] = df[col].fillna("Missing").astype(str)
            continue

        if col in force_numerical:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            numerical_cols.append(col)
            continue

        # --- 2. Automatic Rules ---
        is_object_or_cat = df[col].dtype == "object" or df[col].dtype.name == "category"
        is_binary_flag = col.startswith("hx_") and not col.endswith(("dslast", "dsfirst", "_cnt"))
        is_low_cardinality_int = df[col].nunique() <= 3 and set(df[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})
        is_known_desc = col.endswith("_DESC") or col.endswith("_DES") or col in {"sex", "gender", "ethnicity"}

        if is_object_or_cat or is_binary_flag or is_low_cardinality_int or is_known_desc:
            categorical_cols.append(col)
            df[col] = df[col].fillna("Missing").astype(str)
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            numerical_cols.append(col)

    return df, feature_cols, categorical_cols, numerical_cols


# 2. Apply our clean pipeline
df_clean, columns, categorical, numerical = \
    convert_dtypes_and_classify(df_master,
        force_categorical=[
            #'INDEX_OF_MULTIPLE_DEPRAVATION_DECILE',
            'MAIN_SPECIALTY_CODE',
            'ETNICITY_DES',
            'GENDER_DESC'])

# Designate numerical features to report as Medians [IQR] (nonnormal) vs Means (normal)
# For clinical data, almost all continuous metrics/counts are best represented as nonnormal [IQR]
nonnormal = numerical

# 3. Instantiate TableOne cleanly
mytable = TableOne(
    data=df_clean,
    columns=columns,
    categorical=categorical,
    nonnormal=nonnormal,
    groupby="esbl_label",
    overall=True
)


# 4. Output results and export
output_path = DATA_DIR / "table1_demographics.csv"
mytable.to_csv(output_path)

print(f"\nTable 1 successfully saved to: {output_path.resolve()}")


/usr/local/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,



Table 1 successfully saved to: /app/notebooks/data/synthetic/10000p/table1_demographics.csv


In [15]:
# Show
print(mytable.tabulate(tablefmt="github"))

|                                                           |                                      |   Missing | Overall              | Negative            | Positive             |
|-----------------------------------------------------------|--------------------------------------|-----------|----------------------|---------------------|----------------------|
| n                                                         |                                      |           | 10000                | 8502                | 1498                 |
| AGE_AT_ADMISSION, median [Q1,Q3]                          |                                      |         0 | 56.0 [37.0,76.0]     | 57.0 [37.0,76.0]    | 56.0 [35.0,77.0]     |
| MAIN_SPECIALTY_CODE_DESC, n (%)                           | cardiology                           |           | 2444 (24.4)          | 2080 (24.5)         | 364 (24.3)           |
|                                                           | gastroenterology                 

In [36]:
# -------------------------------------------------------------------
# How to compute scores if needed
# --------------------------------------------------------------------
# Libraries
from pathlib import Path
import pandas as pd

# Local Libraries
from icare_risk.clinphen.scores.gavaghan import compute_gavaghan_score
from icare_risk.clinphen.scores.holmgren import compute_holmgren_score
from icare_risk.clinphen.scores.charlson import compute_charlson_score
from icare_risk.clinphen.scores.jones import compute_jones_score
from icare_risk.clinphen.scores.tumbarello import compute_tumbarello_score
from icare_risk.clinphen.scores.kim import compute_kim_score

# 1. Load your master dataset
DATA_DIR = Path("/app/notebooks/data/synthetic/10000p/")
filename = '03_derived_analysis_dataset.parquet'
df_master = pd.read_parquet(DATA_DIR / filename)

df_master['charlson_score'] = compute_charlson_score(df_master, verbose=0)
df_master['gavaghan_score'] = compute_gavaghan_score(df_master, verbose=0)
df_master['holmgren_score'] = compute_holmgren_score(df_master, verbose=0)
df_master['jones_score'] = compute_jones_score(df_master, verbose=0)
df_master['tumbarello_score'] = compute_tumbarello_score(df_master, verbose=0)
df_master['kim_score'] = compute_kim_score(df_master, verbose=0)

scores = [
    'charlson', 'gavaghan', 'holmgren', 'jones', 'tumbarello', 'kim'
]
for s in scores:
    print("\n")
    print(df_master[f"{s}_score"].value_counts())



charlson_score
1     2742
0     1689
4     1595
2     1374
3     1289
5     1179
6       82
7       31
8        7
10       5
9        4
11       2
13       1
Name: count, dtype: int64


gavaghan_score
0    8916
2    1078
1       6
Name: count, dtype: int64


holmgren_score
0.0    10000
Name: count, dtype: int64


jones_score
0    9608
2     392
Name: count, dtype: int64


tumbarello_score
0    8901
2    1077
4      15
1       6
6       1
Name: count, dtype: int64


kim_score
0    8901
1    1083
5      15
6       1
Name: count, dtype: int64
